# Why does the paper-faithful algorithm (0 inner steps) beat the inner-loop variant? An `INNER_STEPS` ablation

**Motivation.** Section 6.1 found `qmaml_paper` (Algorithm 1, no inner-loop adaptation during pre-training)
clearly beats `qmaml_existing` (the prior repo's inner-loop + first-order-detach variant, `INNER_STEPS=5`).
The paper's Discussion offered a plausible mechanism (the inner loop gives the Learner a noisier training
signal) but explicitly flagged it as untested speculation, recommending "a controlled sweep over
`INNER_STEPS` (0 through, say, 20) holding everything else fixed." This notebook is that sweep.

At `INNER_STEPS=0`, `outer_loop_qmaml`'s inner adaptation loop does nothing (`wT == w0`), so
`w_query = (wT - w0).detach() + w0` collapses to exactly `w0` — mechanically very close to Algorithm 1,
modulo one real difference: `outer_loop_qmaml` always evaluates its loss on the **query** split, while
`pretrain_learner_paper` (Algorithm 1) evaluates on the **support** split. That's a second, independent
variable this ablation does not isolate — noted as a limitation of the ablation itself, not swept away.

In [ ]:
import sys, os, time, json
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "final_model")))

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from config import config
from higgs_data import load_higgs, train_test_split_df, ALL_FEATURE_COLS
from higgs_tasks import make_tasks
from models import TabularFeatureExtractor, PQCModel, HybridModel, freeze_bn
from meta import outer_loop_qmaml

RESULTS_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", "final_model", "results", "inner_steps_ablation"))
os.makedirs(RESULTS_DIR, exist_ok=True)
DATA_PATH = os.path.abspath(os.path.join(os.getcwd(), "..", "final_model", "data_higgs", "higgs_subsample.csv"))
RAW_RESULTS_PATH = os.path.join(RESULTS_DIR, "raw_results.json")
print("results ->", RESULTS_DIR)

## 1. Data and tasks (same as notebook 01b, qubits fixed at 6)

In [ ]:
df = load_higgs(180_000, DATA_PATH)
train_df, test_df = train_test_split_df(df, test_frac=0.2)

meta_tasks = make_tasks(train_df, ALL_FEATURE_COLS, "m_bb", bin_count=6,
                         support_size=8, query_size=8, tasks_per_bin=3, seed=42)
test_meta_tasks = make_tasks(test_df, ALL_FEATURE_COLS, "m_bb", bin_count=6,
                              support_size=8, query_size=8, tasks_per_bin=2, seed=43)
print(f"meta_tasks={len(meta_tasks)}  test_meta_tasks={len(test_meta_tasks)}")

## 2. Sweep `INNER_STEPS` in {0, 1, 2, 5, 10, 20}, 2 seeds, 6 qubits, 10 epochs

If the Discussion's hypothesis is right, performance/convergence should degrade roughly monotonically as
`INNER_STEPS` increases from 0.

In [ ]:
def build_model(num_qubits, depth, seed, feature_cols, train_df):
    torch.manual_seed(seed)
    np.random.seed(seed)
    tab = TabularFeatureExtractor(len(feature_cols), config.CNN_OUTPUT_DIM, num_qubits)
    train_X = torch.from_numpy(train_df[feature_cols].values.astype(np.float32))
    tab.set_norm_stats(train_X.mean(0), train_X.std(0))
    freeze_bn(tab)
    pqc = PQCModel(num_qubits, depth, init_type="zero", bound_angles=True)
    return HybridModel(tab, pqc)

NUM_QUBITS = 6
DEPTH = 2
SEEDS = [0, 1]
EPOCHS = 10
INNER_STEPS_GRID = [0, 1, 2, 5, 10, 20]

config.Q_DEPTH = DEPTH
config.NUM_QUBITS = NUM_QUBITS
config.INNER_LR = 0.02
config.OUTER_LR = 5e-3
config.W0_SCALE = 0.01
config.EPOCHS = EPOCHS

if os.path.isfile(RAW_RESULTS_PATH):
    with open(RAW_RESULTS_PATH) as f:
        all_results = json.load(f)
    print(f"resuming: {len(all_results)} results already on disk")
else:
    all_results = {}

def run_one(inner_steps, seed):
    key = str((inner_steps, seed))
    if key in all_results:
        print(f"skip {key} (already done)")
        return
    print(f"\n=== INNER_STEPS={inner_steps} seed={seed} ===")
    config.INNER_STEPS = inner_steps
    config.CHECKPOINT_DIR = os.path.join(RESULTS_DIR, "ckpt", f"is{inner_steps}_s{seed}")
    model = build_model(NUM_QUBITS, DEPTH, seed, ALL_FEATURE_COLS, train_df)
    t0 = time.time()
    res = outer_loop_qmaml(model, meta_tasks, test_meta_tasks, config.OUTER_LR, True,
                            ckpt_name=f"is{inner_steps}_s{seed}.pth")
    print(f"  done in {time.time()-t0:.1f}s")
    all_results[key] = res
    with open(RAW_RESULTS_PATH, "w") as f:
        json.dump(all_results, f, indent=2)

In [ ]:
run_one(0, 0)

In [ ]:
run_one(0, 1)

In [ ]:
run_one(1, 0)

In [ ]:
run_one(1, 1)

In [ ]:
run_one(2, 0)

In [ ]:
run_one(2, 1)

In [ ]:
run_one(5, 0)

In [ ]:
run_one(5, 1)

In [ ]:
run_one(10, 0)

In [ ]:
run_one(10, 1)

In [ ]:
run_one(20, 0)

In [ ]:
run_one(20, 1)

## 3. Results: does performance degrade monotonically with more inner steps?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
metrics = [("meta_loss", "final meta-loss"), ("val_accuracy", "final val accuracy"), ("gradient_norms", "mean grad norm")]
summary_rows = []
for inner_steps in INNER_STEPS_GRID:
    finals = {"meta_loss": [], "val_accuracy": [], "gradient_norms": []}
    for seed in SEEDS:
        key = str((inner_steps, seed))
        if key not in all_results:
            continue
        r = all_results[key]
        finals["meta_loss"].append(r["meta_loss"][-1])
        finals["val_accuracy"].append(r["val_accuracy"][-1])
        finals["gradient_norms"].append(float(np.mean(r["gradient_norms"])))
    row = {"inner_steps": inner_steps, "n_seeds": len(finals["meta_loss"])}
    for m in finals:
        row[f"{m}_mean"] = float(np.mean(finals[m])) if finals[m] else None
        row[f"{m}_std"] = float(np.std(finals[m])) if finals[m] else None
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(os.path.join(RESULTS_DIR, "summary.csv"), index=False)

for ax, (metric, label) in zip(axes, metrics):
    means = summary_df[f"{metric}_mean"]
    stds = summary_df[f"{metric}_std"]
    ax.errorbar(summary_df["inner_steps"], means, yerr=stds, marker="o", capsize=4)
    ax.set_xlabel("INNER_STEPS")
    ax.set_ylabel(label)
    ax.grid(alpha=0.3)
fig.suptitle(f"INNER_STEPS ablation ({NUM_QUBITS} qubits, {EPOCHS} epochs, {len(SEEDS)} seeds)")
fig.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, "inner_steps_ablation.png"), dpi=150)
plt.show()
summary_df

## 4. Findings (fill in after reviewing the plot/table)

- Is the meta-loss/val-accuracy relationship with `INNER_STEPS` monotonic, as the Discussion's hypothesis
  predicts, or does it saturate/reverse at some point?
- `INNER_STEPS=0` should closely match `qmaml_paper`'s numbers from notebook 01b at 6 qubits (0.364 meta-loss,
  0.578 val acc) modulo the query-vs-support evaluation-split difference noted above -- does it?
- Does gradient norm track the same trend, giving a mechanistic read on *why* (e.g., does more inner-loop
  adaptation destabilize the outer gradient signal, consistent with the Discussion's "noisier training
  signal" hypothesis)?
